# Часть 1. EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

## 1. Загрузка датасета https://www.kaggle.com/uciml/breast-cancer-wisconsin-data

In [ ]:
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
df = pd.read_csv(path + "/data.csv")

## 2. Базовые статистики датасета

In [ ]:
print(df.head())

In [ ]:
print(df.describe())

In [ ]:
print(df.info())

In [ ]:
print(df.isnull().sum())

In [ ]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.drop(columns=['id'], errors='ignore')

## 3. Гистограммы признаков по целевой переменной diagnosis

In [ ]:
features = df.columns[1:-1]

for feature in features:
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x=feature, hue='diagnosis', multiple='stack', kde=True, bins=30)
    plt.title(f'Гистограмма для {feature}')
    plt.xlabel(feature)
    plt.ylabel('Частота')
    plt.legend(title='Диагноз', labels=['Доброкачественный (B)', 'Злокачественный (M)'])
    plt.show()

**Размер опухоли**: Увеличение среднего радиуса, периметра и площади связано с повышенной вероятностью злокачественности опухоли.

**Форма опухоли**: Высокие значения вогнутости и количества вогнутых точек также указывают
на большую вероятность злокачественности.

**Гладкость и текстура**: Эти признаки показывают значительное перекрытие между доброкачественными и злокачественными опухолями, что делает их менее информативными для диагностики.

## 4. heatmap для матрицы корреляций

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
numeric_df = df.select_dtypes(include=['float64', 'int64'])
correlation_matrix = numeric_df.corr()

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt=".1f", cmap='RdBu', square=True, cbar=False)
plt.title('Матрица корреляций')
plt.show()

### Сильная корреляция с размерными признаками:
- **radius_mean, area_mean, perimeter_mean (0.7)**: Корреляция около 0.7 указывает на то, что эти признаки имеют значительное влияние на диагноз опухоли. Это означает, что увеличение радиуса, площади или периметра опухоли связано с повышенной вероятностью злокачественности. Эти признаки могут быть важными для диагностики и классификации опухолей

### Корреляция с формой опухоли:
- **concavity_mean (0.7)** : Сильная положительная корреляция с диагнозом также указывает на то, что вогнутость опухоли может быть важным признаком для различения между доброкачественными и злокачественными опухолями. Более выраженная вогнутость может указывать на более высокий риск злокачественности.
- **concave points_mean (0.8)**: Очень высокая корреляция с диагнозом говорит о том, что количество вогнутых точек на границе опухоли является очень сильным индикатором злокачественности. Это может быть критически важным признаком для диагностики.

### Отсутствие корреляции:
- **fractal_dimension_mean (0.0)**: Нулевая корреляция с диагнозом указывает на то, что этот признак не имеет линейной зависимости с целевой переменной. Это может означать, что фрактальная размерность не является полезным индикатором для различения между доброкачественными и злокачественными опухолями.

Поля **radius_mean, perimeter_mean и area_mean** имеют корреляцию, близкую к 1, это указывает на сильную линейную зависимость между ними. Это может означать, что они предоставляют избыточную информацию, можно сократить эти признаки для упрощения модели и уменьшения риска мультиколлинеарности.

Поля **concavity_mean** и **compactness_mean**, а также **concave points_mean** и **concavity_mean** имеют высокую корреляцию, это также может указывать на избыточность информации




## 5. scatterplot-ы для сильно скоррелированных признаков

In [ ]:
correlated_features = ['radius_mean', 'area_mean', 'perimeter_mean', 'concavity_mean', 'compactness_mean', 'concave points_mean']

df_subset = df[correlated_features + ['diagnosis']]

sns.pairplot(df_subset, hue='diagnosis', palette='RdBu', diag_kind='kde')
plt.suptitle('Парные графики для сильно скоррелированных признаков', y=1.02)
plt.show()

- На scatterplot-ах видно, что точки для **radius_mean, perimeter_mean и area_mean** образуют четкую линейную форму, это подтверждает высокую корреляцию между этими признаками.
- на графиках для **compactness_mean, concave points_mean и concavity_mean** точки распределены более случайным образом, это может указывать на то, что между этими признаками нет четкой линейной зависимости

## 6. boxplots и группировка по целевой переменной

In [ ]:
features_to_plot = df.columns[1:11]
num_features = len(features_to_plot)

ncols = 3
nrows = (num_features + ncols - 1) // ncols

plt.figure(figsize=(15, 5 * nrows))

for i, feature in enumerate(features_to_plot):
    plt.subplot(nrows, ncols, i + 1)
    sns.boxplot(x='diagnosis', y=feature, data=df, hue='diagnosis', palette='RdBu')
    plt.title(f'Boxplot: {feature}')
    plt.xlabel('Diagnosis (0 = Benign, 1 = Malignant)')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

### Вывод

На основе анализа boxplot-ов можно сделать вывод о том что **radius_mean**, **concavity_mean**, **compactness_mean**, **concave points_mean** показывают четкое разделение между злокачественными и доброкачественными опухолями, эти признаки являются наиболее информативными для различения классов.

# Часть 2. Моделирование при помощи kNN

## 1. Разделение данных на train-test, откладываем 30% выборки для тестирования

In [ ]:
from sklearn.model_selection import train_test_split

selected_features = ['radius_mean', 'concavity_mean', 'compactness_mean', 'concave points_mean']

X = df[selected_features]
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f'Размер обучающей выборки: {X_train.shape[0]}')
print(f'Размер тестовой выборки: {X_test.shape[0]}')

## 2. Стандартизация

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print(X_train_scaled.head())

Стандартизация признаков необходима, потому что:

**Разные масштабы**: Признаки могут иметь разные диапазоны значений, что может исказить результаты алгоритмов, основанных на расстояниях (например, kNN).

**Улучшение производительности**: Стандартизация помогает алгоритмам быстрее сходиться и повышает их эффективность.

**Сравнение признаков**: Приведение признаков к одному масштабу упрощает интерпретацию их влияния на целевую переменную.

**Снижение влияния выбросов**: Стандартизация помогает уменьшить влияние выбросов на модель.

### 4. Модель kNN «из коробки»

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc

knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

y_prob = knn.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

print(f'Площадь под ROC-кривой (AUC): {roc_auc:.2f}')

# Визуализация ROC-кривой

plt.figure()
plt.plot(fpr, tpr, color='blue', label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

### 4. Настройка параметра числа соседей на кросс-валидации

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_neighbors': range(1, 11)}
knn = KNeighborsClassifier()

grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train_scaled, y_train)

best_k = grid_search.best_params_['n_neighbors']
best_score = grid_search.best_score_

print(f'Лучшее значение k: {best_k}')
print(f'Лучший F1 Score на кросс-валидации: {best_score:.2f}')

best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_scaled, y_train)

y_pred = best_knn.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')


### Вывод:
В целом, результаты показывают, что модель стала лучше после настройки параметра k=4. Увеличение accuracy и precision указывает на то, что модель более надежна в своих положительных предсказаниях. Однако снижение полноты (recall) может быть проблемой, в контексте задачи нужно минимизировать количество пропущенных положительных случаев (злокачественных новооброзований).

### Изменение порога классификации.

Посмотрим при каком пороге recall становиться максимальным

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure()
plt.plot(thresholds, precision[:-1], label='Precision', color='blue')
plt.plot(thresholds, recall[:-1], label='Recall', color='orange')
plt.xlabel('Порог')
plt.ylabel('Метрики')
plt.title('Precision и Recall в зависимости от порога')
plt.legend()
plt.show()


Видим что при пороге threshold = 0.25 recall становится равным 1, установим новый порог и посмотрим на остальные метрики

In [ ]:
from sklearn.metrics import recall_score


threshold = 0.25
y_pred_adjusted = (y_prob >= threshold).astype(int)

new_recall = recall_score(y_test, y_pred_adjusted)
new_accuracy = accuracy_score(y_test, y_pred_adjusted)
new_precision = precision_score(y_test, y_pred_adjusted)
new_f1 = f1_score(y_test, y_pred_adjusted)

print(f'Recall: {new_recall:.2f}')
print(f'Accuracy: {new_accuracy:.2f}')
print(f'Precision: {new_precision:.2f}')
print(f'F1 Score: {new_f1:.2f}')


В контексте определения злокачественных новообразований метрика recall является критически важной.
Хотя precision снизилась, она все еще может быть приемлемой. Остальные метрики остались на высоком уровне.